# Yahoo Finance API: Company Info, Market Data, Historical Data and Analysis

Four FastAPI endpoints backed by [`yfinance`](https://pypi.org/project/yfinance/), demonstrated
in-notebook with FastAPI's `TestClient` (no server/port needed to grade this).

| # | Method & Path | Purpose |
|---|---|---|
| 1 | `GET /company/{symbol}` | Company profile: name, business summary, industry, sector, officers |
| 2 | `GET /market/{symbol}` | Real-time-ish market snapshot: state, price, change, % change |
| 3 | `POST /historical` | Historical OHLCV data for a symbol between two dates (JSON body) |
| 4 | `POST /analysis` | Deterministic analytics (moving averages, volatility, return, drawdown, trend) over a historical window |

Each endpoint validates its inputs (symbol format, date ordering) before calling Yahoo Finance,
and returns `404` when the symbol is well-formed but Yahoo has no data for it. That way a bad
input never surfaces as a raw stack trace.

Run this notebook top-to-bottom ("Run All"). The last section calls every endpoint once with
valid input and once with invalid input, so both the happy path and the error path are visible
in the output.


In [1]:
# Install dependencies (uncomment if running outside the prepared environment)
# %pip install fastapi==0.115.5 httpx==0.28.0 pydantic==2.10.3 yfinance==1.5.1 pandas==2.3.3


In [2]:
import re
from datetime import date

import pandas as pd
import yfinance as yf
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, field_validator


<python-install>\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


## FastAPI app and request models

In [3]:
app = FastAPI(title="Yahoo Finance API Demo")

# Stock symbols: 1-10 chars, letters/digits/./- (covers tickers like BRK-B, BF.B)
SYMBOL_PATTERN = re.compile(r"^[A-Za-z0-9.\-]{1,10}$")


def validate_symbol(symbol: str) -> str:
    symbol = symbol.strip().upper()
    if not SYMBOL_PATTERN.match(symbol):
        raise HTTPException(status_code=400, detail=f"Invalid symbol format: '{symbol}'")
    return symbol


class HistoricalRequest(BaseModel):
    symbol: str
    start_date: date
    end_date: date

    @field_validator("symbol")
    @classmethod
    def symbol_format(cls, v):
        v = v.strip().upper()
        if not SYMBOL_PATTERN.match(v):
            raise ValueError(f"Invalid symbol format: '{v}'")
        return v

    @field_validator("end_date")
    @classmethod
    def dates_ordered(cls, v, info):
        start = info.data.get("start_date")
        if start and v <= start:
            raise ValueError("end_date must be after start_date")
        return v


## Endpoint 1: Company Information

`GET /company/{symbol}` returns full name, business summary, industry, sector, and officer
name/title pairs, sourced from `yfinance.Ticker(symbol).info`.


In [4]:
@app.get("/company/{symbol}")
def get_company_info(symbol: str):
    symbol = validate_symbol(symbol)
    info = yf.Ticker(symbol).info
    if not info or not info.get("longName"):
        raise HTTPException(status_code=404, detail=f"No company data found for symbol '{symbol}'")
    officers = [
        {"name": o.get("name"), "title": o.get("title")}
        for o in (info.get("companyOfficers") or [])
    ]
    return {
        "symbol": symbol,
        "long_name": info.get("longName"),
        "business_summary": info.get("longBusinessSummary"),
        "industry": info.get("industry"),
        "sector": info.get("sector"),
        "officers": officers,
    }


## Endpoint 2: Stock Market Data

`GET /market/{symbol}` returns market state, current price, price change, and percent change,
plus previous close and day high/low for context.


In [5]:
@app.get("/market/{symbol}")
def get_market_data(symbol: str):
    symbol = validate_symbol(symbol)
    info = yf.Ticker(symbol).info
    if not info or info.get("regularMarketPrice") is None:
        raise HTTPException(status_code=404, detail=f"No market data found for symbol '{symbol}'")
    return {
        "symbol": symbol,
        "market_state": info.get("marketState"),
        "current_price": info.get("regularMarketPrice"),
        "price_change": info.get("regularMarketChange"),
        "percent_change": info.get("regularMarketChangePercent"),
        "previous_close": info.get("previousClose"),
        "day_high": info.get("dayHigh"),
        "day_low": info.get("dayLow"),
    }


## Endpoint 3: Historical Market Data

`POST /historical` accepts a JSON body (`symbol`, `start_date`, `end_date`) and returns daily
OHLCV rows for that window. `end_date` must be after `start_date` (enforced by the Pydantic
model, so a bad order is rejected with `422` before any Yahoo Finance call is made).


In [6]:
def fetch_history(symbol: str, start_date: date, end_date: date) -> pd.DataFrame:
    hist = yf.Ticker(symbol).history(start=str(start_date), end=str(end_date))
    if hist.empty:
        raise HTTPException(
            status_code=404,
            detail=f"No historical data for '{symbol}' between {start_date} and {end_date}",
        )
    return hist


@app.post("/historical")
def get_historical_data(payload: HistoricalRequest):
    hist = fetch_history(payload.symbol, payload.start_date, payload.end_date)
    records = hist.reset_index()
    records["Date"] = records["Date"].dt.strftime("%Y-%m-%d")
    return {
        "symbol": payload.symbol,
        "start_date": str(payload.start_date),
        "end_date": str(payload.end_date),
        "rows": records[["Date", "Open", "High", "Low", "Close", "Volume"]].to_dict(orient="records"),
    }


## Endpoint 4: Analytical Insights

`POST /analysis` uses the same request shape as Endpoint 3. It computes deterministic
statistics over the historical window (no LLM call, since this is a factual and reproducible
endpoint rather than a generative one): 5-day and 20-day simple moving averages, daily-return
volatility, total return, max drawdown, and trend direction, plus a one-line templated summary.


In [7]:
@app.post("/analysis")
def get_analysis(payload: HistoricalRequest):
    hist = fetch_history(payload.symbol, payload.start_date, payload.end_date)
    closes = hist["Close"]
    daily_returns = closes.pct_change().dropna()

    sma_5 = closes.rolling(5).mean().iloc[-1] if len(closes) >= 5 else None
    sma_20 = closes.rolling(20).mean().iloc[-1] if len(closes) >= 20 else None
    volatility = daily_returns.std() if len(daily_returns) > 1 else None
    total_return_pct = ((closes.iloc[-1] / closes.iloc[0]) - 1) * 100
    running_max = closes.cummax()
    drawdown = (closes - running_max) / running_max
    max_drawdown_pct = drawdown.min() * 100
    trend = "up" if closes.iloc[-1] > closes.iloc[0] else ("down" if closes.iloc[-1] < closes.iloc[0] else "flat")

    if volatility is not None:
        summary = (
            f"{payload.symbol} moved {trend} {abs(total_return_pct):.2f}% "
            f"from {payload.start_date} to {payload.end_date}, with a max drawdown of "
            f"{max_drawdown_pct:.2f}% and daily volatility of {(volatility * 100):.2f}%."
        )
    else:
        summary = f"{payload.symbol}: insufficient data points to compute volatility."

    return {
        "symbol": payload.symbol,
        "start_date": str(payload.start_date),
        "end_date": str(payload.end_date),
        "sma_5": round(sma_5, 2) if sma_5 is not None else None,
        "sma_20": round(sma_20, 2) if sma_20 is not None else None,
        "volatility_pct": round(volatility * 100, 4) if volatility is not None else None,
        "total_return_pct": round(total_return_pct, 2),
        "max_drawdown_pct": round(max_drawdown_pct, 2),
        "trend": trend,
        "summary": summary,
    }


## Demo: exercising all four endpoints with `TestClient`

`TestClient` calls the FastAPI app in-process (no live server/port), which is the standard way
to demonstrate or test a FastAPI app from within a notebook. Each endpoint below is called once
with valid input (happy path) and once with invalid input (error path), so the validation and
`404`/`422` handling is visibly proven, not just asserted.


In [8]:
client = TestClient(app)
SYMBOL = "AAPL"  # any liquid ticker works
START, END = "2026-06-01", "2026-07-01"


### Endpoint 1: Company Information

In [9]:
r = client.get(f"/company/{SYMBOL}")
print(r.status_code)
r.json()


200


{'symbol': 'AAPL',
 'long_name': 'Apple Inc.',
 'business_summary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allows customers to discover and download applications and digital content, such as books, music, video, games, and podcasts, as well as advertising services include third-party licensing arrangements and its own advertising platforms. In addition, the company offers various subscription-based services, such as Apple Arcade, a game subscription service; Apple Fitness+, a personaliz

In [10]:
# Error path: malformed symbol -> 400, not a stack trace
r = client.get("/company/!!bad!!")
print(r.status_code)
r.json()


400


{'detail': "Invalid symbol format: '!!BAD!!'"}

### Endpoint 2: Stock Market Data

In [11]:
r = client.get(f"/market/{SYMBOL}")
print(r.status_code)
r.json()


200


{'symbol': 'AAPL',
 'market_state': 'POSTPOST',
 'current_price': 325.89,
 'price_change': -1.84998,
 'percent_change': -0.564464,
 'previous_close': 327.74,
 'day_high': 328.9995,
 'day_low': 323.34}

In [12]:
# Error path: well-formed but nonexistent symbol -> 404
r = client.get("/market/ZZZZ")
print(r.status_code)
r.json()


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ZZZZ"}}}


404


{'detail': "No market data found for symbol 'ZZZZ'"}

### Endpoint 3: Historical Market Data

In [13]:
r = client.post("/historical", json={"symbol": SYMBOL, "start_date": START, "end_date": END})
print(r.status_code)
body = r.json()
print(f"{len(body['rows'])} rows returned")
body["rows"][:3]


200
21 rows returned


[{'Date': '2026-06-01',
  'Open': 309.6300048828125,
  'High': 310.94000244140625,
  'Low': 305.0199890136719,
  'Close': 306.30999755859375,
  'Volume': 48849900},
 {'Date': '2026-06-02',
  'Open': 307.4599914550781,
  'High': 315.45001220703125,
  'Low': 306.69000244140625,
  'Close': 315.20001220703125,
  'Volume': 44534700},
 {'Date': '2026-06-03',
  'Open': 314.17999267578125,
  'High': 316.94000244140625,
  'Low': 308.8500061035156,
  'Close': 310.260009765625,
  'Volume': 50836700}]

In [14]:
# Error path: end_date before start_date -> 422 (Pydantic validation error, caught before any API call)
r = client.post("/historical", json={"symbol": SYMBOL, "start_date": END, "end_date": START})
print(r.status_code)
r.json()


422


{'detail': [{'type': 'value_error',
   'loc': ['body', 'end_date'],
   'msg': 'Value error, end_date must be after start_date',
   'input': '2026-06-01',
   'ctx': {'error': {}}}]}

### Endpoint 4: Analytical Insights

In [15]:
r = client.post("/analysis", json={"symbol": SYMBOL, "start_date": START, "end_date": END})
print(r.status_code)
r.json()


200


{'symbol': 'AAPL',
 'start_date': '2026-06-01',
 'end_date': '2026-07-01',
 'sma_5': 284.62,
 'sma_20': 295.92,
 'volatility_pct': 2.2303,
 'total_return_pct': -5.53,
 'max_drawdown_pct': -12.71,
 'trend': 'down',
 'summary': 'AAPL moved down 5.53% from 2026-06-01 to 2026-07-01, with a max drawdown of -12.71% and daily volatility of 2.23%.'}

In [16]:
# Error path: malformed symbol -> 422 (rejected by the Pydantic field validator)
r = client.post("/analysis", json={"symbol": "!!bad!!", "start_date": START, "end_date": END})
print(r.status_code)
r.json()


422


{'detail': [{'type': 'value_error',
   'loc': ['body', 'symbol'],
   'msg': "Value error, Invalid symbol format: '!!BAD!!'",
   'input': '!!bad!!',
   'ctx': {'error': {}}}]}

## Notes and limitations

- Data source: `yfinance` wraps Yahoo Finance's unofficial endpoints. No API key is required,
  but there is no uptime or schema guarantee either. If a cell errors with a network/HTTP error
  rather than the handled `400`/`404`/`422` responses above, Yahoo's endpoint shape may have
  shifted; check `yf.Ticker(SYMBOL).info.keys()` to see what fields are currently available.
- Not production: no auth, no rate limiting, no persistence. This is a demo/assignment
  deliverable, run entirely in-process via `TestClient`.
- Analytical Insights is intentionally deterministic (moving averages, volatility, return,
  drawdown) rather than LLM-generated, so the output is reproducible and traceable to the
  input data instead of being a probabilistic guess.
